# Rural Bengali Translation Pipeline
### Run each cell in order. Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Install dependencies
!pip install transformers sentencepiece torch pandas openpyxl sacrebleu -q

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Upload your files (corpus_aligned.txt and BanglaRegionalTextCorpus-tnQRMn.xlsx)
# Skip this if you already put them in Google Drive
from google.colab import files
uploaded = files.upload()

In [ ]:
# Cell 4 — Imports and config
import torch
import random
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    AutoModelForSeq2SeqLM, AutoTokenizer,
)
from torch.optim import AdamW

MAX_LEN    = 128
BATCH_SIZE = 8
EPOCHS     = 10
LR         = 2e-5
PATIENCE   = 3
DRIVE_PATH = '/content/drive/MyDrive/bengali_translation'

import os
os.makedirs(DRIVE_PATH, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Cell 5 — Load datasets
def load_pairs(filepath):
    pairs=[]
    with open(filepath,'r',encoding='utf-8') as f:
        for line in f:
            if '|||' in line:
                src,tgt=line.strip().split('|||',1)
                pairs.append((src.strip(),tgt.strip()))
    return pairs

# rural -> standard pairs
r2s_pairs = load_pairs('corpus_aligned.txt')
random.shuffle(r2s_pairs)
r2s_split  = int(0.9*len(r2s_pairs))
r2s_train  = r2s_pairs[:r2s_split]
r2s_val    = r2s_pairs[r2s_split:]

# standard -> english pairs
df = pd.read_excel('BanglaRegionalTextCorpus-tnQRMn.xlsx')
df = df[['Standard_Bangla_Texts','English_Translation']].dropna()
s2e_pairs  = list(zip(df['Standard_Bangla_Texts'].tolist(), df['English_Translation'].tolist()))
random.shuffle(s2e_pairs)
s2e_split  = int(0.9*len(s2e_pairs))
s2e_train  = s2e_pairs[:s2e_split]
s2e_val    = s2e_pairs[s2e_split:]

print(f'R2S  — Train: {len(r2s_train)} | Val: {len(r2s_val)}')
print(f'S2E  — Train: {len(s2e_train)} | Val: {len(s2e_val)}')

In [ ]:
# Cell 6 — Dataset class and train function
class TranslationDataset(Dataset):
    def __init__(self,pairs,tokenizer,model_type='mbart',src_lang=None,tgt_lang=None,prefix=''):
        self.pairs=pairs
        self.tokenizer=tokenizer
        self.model_type=model_type
        self.src_lang=src_lang
        self.tgt_lang=tgt_lang
        self.prefix=prefix

    def __len__(self): return len(self.pairs)

    def __getitem__(self,idx):
        src,tgt=self.pairs[idx]
        src=self.prefix+str(src)
        if self.model_type=='mbart':
            self.tokenizer.src_lang=self.src_lang
            model_inputs=self.tokenizer(src,max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
            with self.tokenizer.as_target_tokenizer():
                labels=self.tokenizer(str(tgt),max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
        else:
            model_inputs=self.tokenizer(src,max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
            labels=self.tokenizer(str(tgt),max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
        model_inputs['labels']=labels['input_ids']
        return {k:v.squeeze(0) for k,v in model_inputs.items()}


def train_model(model, train_loader, val_loader, save_path):
    optimizer=AdamW(model.parameters(),lr=LR)
    best_val_loss=float('inf')
    no_improve=0
    os.makedirs(save_path,exist_ok=True)
    for epoch in range(1,EPOCHS+1):
        model.train()
        total_loss=0
        for i,batch in enumerate(train_loader):
            batch={k:v.to(device) for k,v in batch.items()}
            loss=model(**batch).loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step()
            total_loss+=loss.item()
            if i%10==0:
                print(f'  Epoch {epoch} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}')
        model.eval()
        val_loss=0
        with torch.no_grad():
            for batch in val_loader:
                batch={k:v.to(device) for k,v in batch.items()}
                val_loss+=model(**batch).loss.item()
        avg_val=val_loss/len(val_loader)
        print(f'Epoch {epoch} | Train: {total_loss/len(train_loader):.4f} | Val: {avg_val:.4f}')
        if avg_val<best_val_loss:
            best_val_loss=avg_val
            no_improve=0
            model.save_pretrained(save_path)
            print(f'  --> Best saved to Drive! (val: {best_val_loss:.4f})\n')
        else:
            no_improve+=1
            print(f'  --> No improvement ({no_improve}/{PATIENCE})\n')
            if no_improve>=PATIENCE:
                print(f'Early stopping at epoch {epoch}')
                break

## Rural → Standard Bengali

In [ ]:
# Cell 7 — mBART rural -> standard
print('=== mBART-50: Rural -> Standard ===')
mbart_tok   = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')
mbart_model = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50-many-to-many-mmt').to(device)
train_loader = DataLoader(TranslationDataset(r2s_train,mbart_tok,'mbart','en_XX','en_XX'),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(TranslationDataset(r2s_val,  mbart_tok,'mbart','en_XX','en_XX'),batch_size=BATCH_SIZE)
train_model(mbart_model,train_loader,val_loader,f'{DRIVE_PATH}/mbart_r2s')
del mbart_model; torch.cuda.empty_cache()

In [ ]:
# Cell 8 — mT5 rural -> standard
print('=== mT5: Rural -> Standard ===')
mt5_tok   = AutoTokenizer.from_pretrained('google/mt5-small')
mt5_model = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small').to(device)
train_loader = DataLoader(TranslationDataset(r2s_train,mt5_tok,'t5',prefix='normalize dialect: '),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(TranslationDataset(r2s_val,  mt5_tok,'t5',prefix='normalize dialect: '),batch_size=BATCH_SIZE)
train_model(mt5_model,train_loader,val_loader,f'{DRIVE_PATH}/mt5_r2s')
del mt5_model; torch.cuda.empty_cache()

In [ ]:
# Cell 9 — BanglaT5 rural -> standard
print('=== BanglaT5: Rural -> Standard ===')
bt5_tok   = AutoTokenizer.from_pretrained('csebuetnlp/banglat5')
bt5_model = AutoModelForSeq2SeqLM.from_pretrained('csebuetnlp/banglat5').to(device)
train_loader = DataLoader(TranslationDataset(r2s_train,bt5_tok,'t5',prefix='normalize dialect: '),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(TranslationDataset(r2s_val,  bt5_tok,'t5',prefix='normalize dialect: '),batch_size=BATCH_SIZE)
train_model(bt5_model,train_loader,val_loader,f'{DRIVE_PATH}/banglat5_r2s')
del bt5_model; torch.cuda.empty_cache()

## Standard Bengali → English

In [ ]:
# Cell 10 — mBART standard -> english
print('=== mBART-50: Standard -> English ===')
mbart_tok   = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')
mbart_model = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50-many-to-many-mmt').to(device)
train_loader = DataLoader(TranslationDataset(s2e_train,mbart_tok,'mbart','bn_IN','en_XX'),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(TranslationDataset(s2e_val,  mbart_tok,'mbart','bn_IN','en_XX'),batch_size=BATCH_SIZE)
train_model(mbart_model,train_loader,val_loader,f'{DRIVE_PATH}/mbart_s2e')
del mbart_model; torch.cuda.empty_cache()

In [ ]:
# Cell 11 — mT5 standard -> english
print('=== mT5: Standard -> English ===')
mt5_tok   = AutoTokenizer.from_pretrained('google/mt5-small')
mt5_model = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small').to(device)
train_loader = DataLoader(TranslationDataset(s2e_train,mt5_tok,'t5',prefix='translate Bengali to English: '),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(TranslationDataset(s2e_val,  mt5_tok,'t5',prefix='translate Bengali to English: '),batch_size=BATCH_SIZE)
train_model(mt5_model,train_loader,val_loader,f'{DRIVE_PATH}/mt5_s2e')
del mt5_model; torch.cuda.empty_cache()

In [ ]:
# Cell 12 — BanglaT5 standard -> english
print('=== BanglaT5: Standard -> English ===')
bt5_tok   = AutoTokenizer.from_pretrained('csebuetnlp/banglat5')
bt5_model = AutoModelForSeq2SeqLM.from_pretrained('csebuetnlp/banglat5').to(device)
train_loader = DataLoader(TranslationDataset(s2e_train,bt5_tok,'t5',prefix='translate Bengali to English: '),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(TranslationDataset(s2e_val,  bt5_tok,'t5',prefix='translate Bengali to English: '),batch_size=BATCH_SIZE)
train_model(bt5_model,train_loader,val_loader,f'{DRIVE_PATH}/banglat5_s2e')
del bt5_model; torch.cuda.empty_cache()

## Done! All models saved to Google Drive under `bengali_translation/`